In [1]:
!pip install albumentations==1.3.0 opencv-python-headless


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

import os
import random
import shutil
from pathlib import Path
import cv2
import albumentations as A

# مسیرها
images_folder = "/content/drive/MyDrive/New folder (3)/images"
labels_folder = "/content/drive/MyDrive/New folder (3)/labels"
output_folder = "/content/dataset_yolo5"

train_ratio = 0.8  # نسبت train/val

# Augmentation ها برای train
transform = A.Compose([
    A.HorizontalFlip(p=0.5),                     # قرینه افقی
    A.RandomBrightnessContrast(p=0.5),           # تغییر نور و کنتراست
    A.GaussNoise(p=0.2),                         # نویز Gaussian
    A.Blur(p=0.2),                               # بلور ملایم
    A.ShiftScaleRotate(shift_limit=0.05,
                       scale_limit=0.1,
                       rotate_limit=15, p=0.7)   # چرخش ±15 درجه (طبیعی)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


# ساخت پوشه‌ها
for split in ["train", "val"]:
    os.makedirs(f"{output_folder}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_folder}/labels/{split}", exist_ok=True)

# لیست تصاویر
images = [f for f in os.listdir(images_folder) if f.endswith(('.jpg', '.png'))]
random.shuffle(images)

train_count = int(len(images) * train_ratio)
train_images = images[:train_count]
val_images = images[train_count:]

def process_image(image_name, split, augment=False, aug_count=2):
    img_path = os.path.join(images_folder, image_name)
    lbl_path = os.path.join(labels_folder, Path(image_name).stem + ".txt")

    img = cv2.imread(img_path)

    # خواندن لیبل‌ها
    bboxes, class_labels = [], []
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f.readlines():
                cls, x, y, bw, bh = map(float, line.strip().split())
                bboxes.append([x, y, bw, bh])
                class_labels.append(int(cls))

    # ذخیره نسخه اصلی
    shutil.copy(img_path, os.path.join(output_folder, "images", split, image_name))
    shutil.copy(lbl_path, os.path.join(output_folder, "labels", split, Path(image_name).stem + ".txt"))

    # ساخت نسخه‌های augment فقط برای train
    if augment and len(bboxes) > 0:
        for i in range(aug_count):
            augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
            img_aug = augmented['image']
            bboxes_aug = augmented['bboxes']
            class_labels_aug = augmented['class_labels']

            new_name = f"{Path(image_name).stem}_aug{i}.jpg"
            cv2.imwrite(os.path.join(output_folder, "images", split, new_name), img_aug)

            with open(os.path.join(output_folder, "labels", split, f"{Path(new_name).stem}.txt"), 'w') as f:
                for bbox, cls in zip(bboxes_aug, class_labels_aug):
                    x, y, bw, bh = bbox
                    f.write(f"{cls} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

# اجرای روی train و val
for img_file in train_images:
    process_image(img_file, "train", augment=True, aug_count=3)  # هر تصویر train → 3 نسخه augment
for img_file in val_images:
    process_image(img_file, "val", augment=False)

print("✅ دیتاست YOLOv5 آماده شد همراه با Augmentations اصولی")

# ساخت فایل data.yaml
classes = ["marziye", "one", "two"]  # 🔴 اینجا باید اسم کلاس‌های واقعی‌تو بذاری
with open(os.path.join(output_folder, "data.yaml"), "w") as f:
    f.write(f"train: {output_folder}/images/train\n")
    f.write(f"val: {output_folder}/images/val\n\n")
    f.write(f"nc: {len(classes)}\n")
    f.write(f"names: {classes}\n")


✅ دیتاست YOLOv5 آماده شد همراه با Augmentations اصولی


In [9]:
import os

train_path = "/content/dataset_yolo5/images/train"  # مسیر پوشه train
train_images = [f for f in os.listdir(train_path) if f.lower().endswith(('.jpg', '.png'))]

print("📸 تعداد عکس‌های train:", len(train_images))


📸 تعداد عکس‌های train: 256


In [4]:
!git clone https://github.com/ultralytics/yolov5


Cloning into 'yolov5'...
remote: Enumerating objects: 17564, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 17564 (delta 33), reused 7 (delta 7), pack-reused 17510 (from 3)
Receiving objects: 100% (17564/17564), 16.65 MiB | 17.24 MiB/s, done.
Resolving deltas: 100% (12035/12035), done.


In [5]:
%cd yolov5


/content/yolov5


In [6]:
!pip install -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.5 MB/s eta 0:00:00


In [7]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data /content/dataset_yolo5/data.yaml \
  --weights yolov5l.pt \
  --name final_tunedv44 \
  --patience 50



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-09-05 11:28:09.966966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757071689.986996    2866 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757071689.993505    2866 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS wh

In [ ]:
%cd /content/yolov5


/content/yolov5


In [8]:
!python detect.py \
    --weights /content/yolov5/runs/train/final_tunedv44/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --iou-thres 0.45 \
    --source /content/drive/MyDrive/testtt.mp4







detect: weights=['/content/yolov5/runs/train/final_tunedv44/weights/best.pt'], source=/content/drive/MyDrive/testtt.mp4, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-430-g459d8bf0 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 267 layers, 46119048 parameters, 0 gradients, 107.7 GFLOPs
video 1/1 (1/426) /content/drive/MyDrive/testtt.mp4: 384x640 1 marziye, 1 two, 54.1ms
video 1/1 (2/426) /content/drive/MyDrive/testtt.mp4: 384x640 1 marziye, 1 two, 31.6ms
video 1/1 (3/426) /content/drive/MyDrive/testtt.mp4: 384x640 1 marziye, 1 two, 31.5ms
video 1/1 (4/